# Exp1 : **Data Pre-processing**


## **Setup & Load Dataset**

In [21]:
import pandas as pd
import numpy as np

# Load CSV into DataFrame
df = pd.read_csv("data.csv") # Using Customer Dataset

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()


Shape: (17, 5)
Columns: ['Age', 'Income', 'Gender', 'City', 'Purchased']


,Age,Income,Gender,City,Purchased
0,25.0,50000.0,Male,Delhi,Yes
1,30.0,60000.0,Female,Mumbai,No
2,NaN,52000.0,Female,Delhi,Yes
3,40.0,80000.0,NaN,Chennai,No
4,35.0,NaN,Male,Mumbai,Yes


## Explore Dataset

In [22]:
print("\nShape:", df.shape)
print("\nColumns:", df.columns.tolist())


Shape: (17, 5)

Columns: ['Age', 'Income', 'Gender', 'City', 'Purchased']


In [14]:
# Inspect data types and summary
print("Data types:\n", df.dtypes)
print("\nSummary statistics:\n", df.describe().T)

# Check missing values
print("\nMissing values per column:\n", df.isnull().sum())


Data types:
 Age          float64
Income       float64
Gender        object
City          object
Purchased     object
dtype: object

Summary statistics:
         count     mean           std      min      25%      50%      75%  \
Age      15.0     35.8      7.848567     25.0     29.5     35.0     41.0   
Income   16.0  71312.5  15460.568122  50000.0  59750.0  68000.0  81250.0   

             max  
Age         50.0  
Income  100000.0  

Missing values per column:
 Age          2
Income       1
Gender       2
City         1
Purchased    0
dtype: int64


## **Handle Missing Values**

In [23]:
from sklearn.impute import SimpleImputer # For missing value imputation
# print("\nBefore Missing Value Imputation:\n", df)
print(df.isna().sum()) # Check missing values again

# Numeric: mean imputation
num_imputer = SimpleImputer(strategy="mean")
df["Age"] = num_imputer.fit_transform(df[["Age"]])
df["Income"] = num_imputer.fit_transform(df[["Income"]])

# Categorical: mode imputation
cat_imputer = SimpleImputer(strategy="most_frequent")
df["Gender"] = cat_imputer.fit_transform(df[["Gender"]])[:, 0]
df["City"]   = cat_imputer.fit_transform(df[["City"]])[:, 0]

# print("\nAfter Missing Value Imputation:\n", df)
print(df.isna().sum()) # Check missing values again

Age          2
Income       1
Gender       2
City         1
Purchased    0
dtype: int64
Age          0
Income       0
Gender       0
City         0
Purchased    0
dtype: int64


Numeric columns (`Age`, `Income`):

- Both columns had missing values (`NaN`).
- The mean imputer replaced missing values with the mean of each column.
    - Example:  
      - Missing `Age` → replaced with `35.8` (mean of known ages)
      - Missing `Income` → replaced with `71,312.5` (mean of known incomes)
- That's why you see `35.8` and `71,312.5` in the filled table.

Categorical columns (`Gender`, `City`):

- These columns also had missing values.
- The most frequent imputer (mode) filled them with the most common category:
    - `Gender`: replaced missing with `"Female"` (most frequent)
    - `City`: replaced missing with `"Delhi"` (most frequent)
- The output looks "normal" because imputation replaced blanks with real values that blend in (e.g., `"Female"` or `"Delhi"`).
- The only obvious numeric evidence is where `NaN` turned into `35.8` and `71,312.5`.


## **Encode Categorical Variables**

In [24]:
from sklearn.preprocessing import LabelEncoder
# from sklearn.preprocessing import OneHotEncoder

# Label encode target variable
le = LabelEncoder()
df["Purchased"] = le.fit_transform(df["Purchased"])  # Yes=1, No=0

# One-hot encode nominal categorical variables
df = pd.get_dummies(df, columns=["Gender", "City"], drop_first=True)
print("\nAfter Encoding:\n", df)


After Encoding:
      Age    Income  Purchased  Gender_Male  City_Delhi  City_Mumbai
0   25.0   50000.0          1         True        True        False
1   30.0   60000.0          0        False       False         True
2   35.8   52000.0          1        False        True        False
3   40.0   80000.0          0        False       False        False
4   35.0   71312.5          1         True       False         True
5   50.0  100000.0          1         True        True        False
6   35.8   75000.0          0        False        True        False
7   28.0   62000.0          1        False       False        False
8   45.0   90000.0          0         True       False         True
9   33.0   58000.0          1        False        True        False
10  29.0   61000.0          0        False       False         True
11  38.0   72000.0          1         True       False        False
12  42.0   85000.0          0        False        True        False
13  31.0   64000.0          1 

## **Detect & Handle Outliers**

In [25]:
# Outlier detection using IQR method
Q1 = df["Income"].quantile(0.25)
Q3 = df["Income"].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
df = df[(df["Income"] >= lower) & (df["Income"] <= upper)]
print("\nAfter Outlier Removal:\n", df)



After Outlier Removal:
      Age    Income  Purchased  Gender_Male  City_Delhi  City_Mumbai
0   25.0   50000.0          1         True        True        False
1   30.0   60000.0          0        False       False         True
2   35.8   52000.0          1        False        True        False
3   40.0   80000.0          0        False       False        False
4   35.0   71312.5          1         True       False         True
5   50.0  100000.0          1         True        True        False
6   35.8   75000.0          0        False        True        False
7   28.0   62000.0          1        False       False        False
8   45.0   90000.0          0         True       False         True
9   33.0   58000.0          1        False        True        False
10  29.0   61000.0          0        False       False         True
11  38.0   72000.0          1         True       False        False
12  42.0   85000.0          0        False        True        False
13  31.0   64000.0     

# What is IQR?
- IQR (Interquartile Range) = Q3 – Q1
- Q1 = 25th percentile (value below which 25% of data falls)
- Q3 = 75th percentile (value below which 75% of data falls)
- So, IQR = the spread of the middle 50% of the data.

🔹 What the code does:
- lower = Q1 - 1.5*IQR
- upper = Q3 + 1.5*IQR
- df = df[(df["Income"] >= lower) & (df["Income"] <= upper)]


It defines lower bound and upper bound for acceptable values.
Any value below lower or above upper is considered an outlier.
Then it removes those rows from the dataset.

🔹 Why use IQR for outliers?

Outliers are extreme values that can skew averages and make models unstable.
IQR rule (Tukey’s method) is a robust statistical method because it’s based on percentiles, not mean/SD (which are sensitive to outliers).
Works well for numerical data like Income, Age, Salary, etc.

🔹 In your case (Income)
- Suppose:
- Q1 = 60000 , Q3 = 90000
- IQR = 30000
- Bounds: Lower = 60000 – (1.5×30000) = 15000
- Upper = 90000 + (1.5×30000) = 135000

✅ So, any Income outside [15000, 135000] will be removed.

- In your dataset, all incomes (50000–100000) are within this range → no rows get removed.

## **Balance Dataset**

In [ ]:
from sklearn.utils import resample
print("\nBefore Balancing:\n", df["Purchased"].value_counts())

# Split majority and minority classes
majority = df[df["Purchased"] == df["Purchased"].value_counts().idxmax()]
minority = df[df["Purchased"] == df["Purchased"].value_counts().idxmin()]

if len(majority) != len(minority):
    minority_upsampled = resample(minority, 
                                  replace=True, 
                                  n_samples=len(majority), 
                                  random_state=42)
    df = pd.concat([majority, minority_upsampled])
print("\nAfter Balancing:\n", df["Purchased"].value_counts())

# Benefits of Balancing
# 1. Prevents the model from always predicting the majority class.
# 2. Ensures fair learning from both classes.
# 3. Especially important for fraud detection, medical diagnosis, etc.


Before Balancing:
 Purchased
1    9
0    8
Name: count, dtype: int64

After Balancing:
 Purchased
1    9
0    9
Name: count, dtype: int64


## **Transform Skewed Features**

In [27]:
# Identify numeric columns (excluding bool and target)
num_cols = df.columns[df.dtypes != "bool"].drop("Purchased")

# Check skewness
skewness = df[num_cols].skew().sort_values(key=lambda x: abs(x), ascending=False)
print(skewness.head(10))

# Apply log1p to highly skewed features
skewed_feats = skewness[abs(skewness) > 1].index
for col in skewed_feats:
    df[col] = np.log1p(df[col] - df[col].min() + 1)

print("Applied log transform to:", skewed_feats.tolist())


Income    1.063309
Age       1.018321
dtype: float64
Applied log transform to: ['Income', 'Age']


## **Scale Features**

In [ ]:
from sklearn.preprocessing import StandardScaler

X = df.drop("Purchased", axis=1)
y = df["Purchased"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X = pd.DataFrame(X_scaled, columns=X.columns)
X["Purchased"] = y.values

X.head()

# what it does:
# It standardizes features by removing the mean and scaling to unit variance.

,Age,Income,Gender_Male,City_Delhi,City_Mumbai,Purchased
0,-2.078147,-3.870288,1.253566,1.414214,-0.797724,1
1,0.571681,-0.692186,-0.797724,1.414214,-0.797724,1
2,0.479554,0.395851,1.253566,-0.707107,1.253566,1
3,1.637140,0.788091,1.253566,1.414214,-0.797724,1
4,-0.770161,0.131660,-0.797724,-0.707107,-0.797724,1


## **Train-Test Split**

In [29]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X.drop("Purchased", axis=1),
    X["Purchased"],
    test_size=0.2, # 20% test size (80-20 split)
    stratify=X["Purchased"],
    random_state=42 
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train class distribution:\n", y_train.value_counts())
print("Test class distribution:\n", y_test.value_counts())


Train shape: (14, 5) Test shape: (4, 5)
Train class distribution:
 Purchased
1    7
0    7
Name: count, dtype: int64
Test class distribution:
 Purchased
0    2
1    2
Name: count, dtype: int64


In [30]:
# Save cleaned + scaled customer dataset
X.to_csv("customer_cleaned_scaled.csv", index=False)
print("✅ Saved customer_cleaned_scaled.csv with shape:", X.shape)
print("First 5 rows of the cleaned & scaled customer dataset:")
X.head()

✅ Saved customer_cleaned_scaled.csv with shape: (18, 6)
First 5 rows of the cleaned & scaled customer dataset:


,Age,Income,Gender_Male,City_Delhi,City_Mumbai,Purchased
0,-2.078147,-3.870288,1.253566,1.414214,-0.797724,1
1,0.571681,-0.692186,-0.797724,1.414214,-0.797724,1
2,0.479554,0.395851,1.253566,-0.707107,1.253566,1
3,1.637140,0.788091,1.253566,1.414214,-0.797724,1
4,-0.770161,0.131660,-0.797724,-0.707107,-0.797724,1
